# Bicycle Accident Analysis — Paderborn 2024

This notebook analyses bicycle accident data from the **Destatis Unfallatlas** and enriches it with OpenStreetMap (OSM) data (schools and streetlights) to explore spatial patterns of accidents in Paderborn.

In [2]:
!pip install folium requests pandas -q

import requests
import pandas as pd
import folium


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: C:\Users\MoZa\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 1. Load accident data

Read the CSV from Destatis. The file uses **semicolons** as separators (German format), so we pass `sep=';'`.

In [3]:
path = r"C:\Users\MoZa\OneDrive - Universität Paderborn\0_UPB\6\Data Viz\csv\Unfallorte2024_LinRef.csv"

bike_acci = pd.read_csv(path, sep=';')


C:\Users\MoZa\AppData\Local\Temp\ipykernel_26604\3345210901.py:3: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  bike_acci = pd.read_csv(path, sep=';')


## 2. Explore the raw data

Check column names, data types, and a preview to understand the structure before any cleaning.

In [4]:
# Always do this before cleaning — understand what you have
print(bike_acci.columns)

print(bike_acci.dtypes)

Index(['OID_', 'UIDENTSTLAE', 'ULAND', 'UREGBEZ', 'UKREIS', 'UGEMEINDE',
       'UJAHR', 'UMONAT', 'USTUNDE', 'UWOCHENTAG', 'UKATEGORIE', 'UART',
       'UTYP1', 'ULICHTVERH', 'IstStrassenzustand', 'IstRad', 'IstPKW',
       'IstFuss', 'IstKrad', 'IstGkfz', 'IstSonstige', 'LINREFX', 'LINREFY',
       'XGCSWGS84', 'YGCSWGS84', 'PLST'],
      dtype='object')
OID_                   int64
UIDENTSTLAE           object
ULAND                  int64
UREGBEZ                int64
UKREIS                 int64
UGEMEINDE              int64
UJAHR                  int64
UMONAT                 int64
USTUNDE                int64
UWOCHENTAG             int64
UKATEGORIE             int64
UART                   int64
UTYP1                  int64
ULICHTVERH             int64
IstStrassenzustand     int64
IstRad                 int64
IstPKW                 int64
IstFuss                int64
IstKrad                int64
IstGkfz                int64
IstSonstige            int64
LINREFX               object
LIN

In [5]:
pd.set_option('display.max_columns', None)
bike_acci.head()


,OID_,UIDENTSTLAE,ULAND,UREGBEZ,UKREIS,UGEMEINDE,UJAHR,UMONAT,USTUNDE,UWOCHENTAG,UKATEGORIE,UART,UTYP1,ULICHTVERH,IstStrassenzustand,IstRad,IstPKW,IstFuss,IstKrad,IstGkfz,IstSonstige,LINREFX,LINREFY,XGCSWGS84,YGCSWGS84,PLST
0,1,1240526125013102024,1,0,59,44,2024,5,23,1,3,1,5,2,0,0,1,0,0,0,0,"525162,376299999654293","6045497,205000000074506","9,389075627000068","54,556379612000057",1
1,2,1240526105132502024,1,0,62,94,2024,5,23,1,3,8,6,2,1,0,1,0,0,0,0,"600576,090200000442564","5964744,324200000613928","10,527878375000057","53,821498344000076",1
2,3,1240525171013972024,1,0,60,88,2024,5,18,7,3,9,1,0,0,0,0,0,1,0,0,"574734,079499999992549","5969976,972100000828505","10,136678418000031","53,872878104000051",1
3,4,1240525171013602024,1,0,60,47,2024,5,16,7,3,5,3,0,0,1,1,0,0,0,0,"567578,579099999740720","5963041,894099999219179","10,026340679000043","53,811535772000070",1
4,5,1240525171013522024,1,0,60,4,2024,5,19,7,3,5,2,0,0,0,1,0,0,0,0,"561431,042000000365078","5973289,735400000587106","9,935041162000061","53,904394936000074",1


## 3. Clean and standardise coordinates

The coordinate columns (`XGCSWGS84`, `YGCSWGS84`) use German decimal commas. We rename them to `lat`/`lon` and convert to proper floats.

In [6]:
# Rename coordinate columns to standard names
bike_acci = bike_acci.rename(columns={
    "XGCSWGS84": "lon",
    "YGCSWGS84": "lat"
})

# Fix decimal separator (German CSVs use comma instead of dot)
bike_acci["lat"] = bike_acci["lat"].astype(str).str.replace(",", ".").astype(float)
bike_acci["lon"] = bike_acci["lon"].astype(str).str.replace(",", ".").astype(float)

# Filter: only bicycle accidents
bike_acci = bike_acci[bike_acci["IstRad"] == 1].copy()
print(f"Total accidents: {len(bike_acci)}")
print(f"Bicycle accidents: {len(bike_acci)}")

# Filter: only Paderborn bounding box (rough coordinates)
# Paderborn is roughly between these coordinates
lat_min, lat_max = 51.65, 51.78
lon_min, lon_max = 8.65, 8.85

bike_acci = bike_acci[
    (bike_acci["lat"] >= lat_min) & (bike_acci["lat"] <= lat_max) &
    (bike_acci["lon"] >= lon_min) & (bike_acci["lon"] <= lon_max)
].copy()

print(f"Bicycle accidents in Paderborn: {len(bike_acci)}")
bike_acci.head()


Total accidents: 86543
Bicycle accidents: 86543
Bicycle accidents in Paderborn: 246


,OID_,UIDENTSTLAE,ULAND,UREGBEZ,UKREIS,UGEMEINDE,UJAHR,UMONAT,USTUNDE,UWOCHENTAG,UKATEGORIE,UART,UTYP1,ULICHTVERH,IstStrassenzustand,IstRad,IstPKW,IstFuss,IstKrad,IstGkfz,IstSonstige,LINREFX,LINREFY,lon,lat,PLST
25844,25845,5240112411132780673,5,7,74,32,2024,1,14,6,3,0,3,0,1,1,0,0,1,0,0,"480973,502999999560416","5731161,901000000536442",8.724498,51.731054,1
26916,26917,5240203411232530955,5,7,74,32,2024,2,8,7,3,7,1,0,1,1,0,0,0,0,0,"480331,287800000049174","5735749,133099999278784",8.714939,51.772277,1
27912,27913,5240123411232117788,5,7,74,32,2024,1,7,3,2,5,3,1,0,1,1,0,0,0,0,"480184,146900000050664","5736187,297299999743700",8.712782,51.776211,1
30309,30310,5240212411232825939,5,7,74,32,2024,2,15,2,3,0,1,0,1,1,0,0,0,0,0,"476666,576999999582767","5734099,529999999329448",8.661938,51.757304,1
30722,30723,5240318411132052357,5,7,74,32,2024,3,8,2,3,3,6,0,0,1,0,0,0,0,1,"484280,855000000447035","5733726,371999999508262",8.772272,51.754215,1


## 4. Add readable labels

Map numeric codes (severity, light conditions, weekday) to human-readable strings so charts and popups are self-explanatory.

In [7]:
bike_acci["severity"] = bike_acci["UKATEGORIE"].map({
    1: "Fatal", 2: "Serious injury", 3: "Light injury"
})
bike_acci["light"] = bike_acci["ULICHTVERH"].map({
    0: "Daylight", 1: "Dusk/dawn", 2: "Dark"
})
bike_acci["weekday"] = bike_acci["UWOCHENTAG"].map({
    1: "Sunday", 2: "Monday", 3: "Tuesday",
    4: "Wednesday", 5: "Thursday", 6: "Friday", 7: "Saturday"
})

## 6. Fetch OpenStreetMap data

Query the **Overpass API** to get locations of schools and streetlights within Paderborn's administrative boundary. Results are returned as a clean DataFrame with `lat`, `lon`, and `name`.

In [8]:
def fetch_osm(query):
    """Send an Overpass QL query and return a clean DataFrame."""
    headers = {"User-Agent": "DataVis-Assignment/1.0 (student project)"}
    for url in [
        "https://overpass-api.de/api/interpreter",
        "https://overpass.kumi.systems/api/interpreter",
    ]:
        try:
            response = requests.post(url, data={"data": query}, headers=headers, timeout=60)
            response.raise_for_status()
            break
        except Exception as e:
            last_error = e
    else:
        raise last_error

    elements = response.json()["elements"]
    rows = []
    for el in elements:
        lat = el.get("lat") or el.get("center", {}).get("lat")
        lon = el.get("lon") or el.get("center", {}).get("lon")
        if lat and lon:
            row = {"lat": lat, "lon": lon}
            row.update(el.get("tags", {}))
            rows.append(row)

    return pd.DataFrame(rows)

### 6a. Fetch schools in Paderborn

In [9]:
school_query = """
[out:json][timeout:60];
area["name"="Paderborn"]["boundary"="administrative"]->.city;
(
  node["amenity"="school"](area.city);
  way["amenity"="school"](area.city);
);
out center tags;
"""

df_schools = fetch_osm(school_query)
df_schools["type"] = "school"
df_schools = df_schools[["lat", "lon", "name", "type"]].fillna("Unknown")

# Remove duplicates: OSM returns both a node and a way for the same school
df_schools = df_schools.drop_duplicates(subset="name")

# Keep only genuine educational institutions by name keyword
# (removes mis-tagged OSM entries like art studios etc.)
school_keywords = [
    "schule", "gymnasium", "berufskolleg", "realschule", "hauptschule",
    "gesamtschule", "förderschule", "waldorf", "bildung", "kolleg",
    "internat", "kita", "kindergarten", "akademie"
]
mask = df_schools["name"].str.lower().str.contains("|".join(school_keywords), na=False)
df_schools = df_schools[mask].reset_index(drop=True)

print(f"Found {len(df_schools)} schools")


Found 52 schools


### 6b. Fetch streetlights

In [10]:
light_query = """
[out:json][timeout:60];
area["name"="Paderborn"]["boundary"="administrative"]->.city;
(
  node["highway"="street_lamp"](area.city);
);
out center;
"""

df_lights = fetch_osm(light_query)
df_lights["type"] = "streetlight"
df_lights["name"] = "Street lamp"

df_lights = df_lights[["lat", "lon", "name", "type"]].fillna("Unknown")
print(f"Found {len(df_lights)} streetlights")
df_lights.head()

Found 2258 streetlights


,lat,lon,name,type
0,51.710010,8.767271,Street lamp,streetlight
1,51.710247,8.767560,Street lamp,streetlight
2,51.708053,8.771479,Street lamp,streetlight
3,51.707682,8.771767,Street lamp,streetlight
4,51.707661,8.771783,Street lamp,streetlight


## 5. Flag accidents near schools

Mark each accident as near a school if it falls within ~200 m (≈ 0.002°) of any school location. This helper function does a simple bounding-box check.

In [11]:
# Flag accidents within ~200m using a coordinate box
OFFSET = 0.002  # roughly 200 meters in degrees

def is_near_school_simple(acc_lat, acc_lon, df_schools):
    return any(
        (abs(acc_lat - s["lat"]) < OFFSET) and 
        (abs(acc_lon - s["lon"]) < OFFSET)
        for _, s in df_schools.iterrows()
    )

bike_acci["near_school"] = bike_acci.apply(
    lambda row: is_near_school_simple(row["lat"], row["lon"], df_schools),
    axis=1
)

### 6c. Combine schools and streetlights

Merge both OSM datasets into one DataFrame for easy filtering and plotting.

In [12]:
df_osm = pd.concat([df_schools, df_lights], ignore_index=True)
print(df_osm["type"].value_counts())
df_osm.tail(10)

type
streetlight    2258
school           52
Name: count, dtype: int64


,lat,lon,name,type
2300,51.714666,8.752595,Street lamp,streetlight
2301,51.714580,8.748041,Street lamp,streetlight
2302,51.717046,8.746662,Street lamp,streetlight
2303,51.706203,8.723716,Street lamp,streetlight
2304,51.706324,8.723855,Street lamp,streetlight
2305,51.693990,8.681384,Street lamp,streetlight
2306,51.693758,8.681875,Street lamp,streetlight
2307,51.693808,8.681566,Street lamp,streetlight
2308,51.694133,8.681610,Street lamp,streetlight
2309,51.714001,8.741228,Street lamp,streetlight


## 7. Interactive map

Plot schools (blue markers) and streetlights (orange circles) on an interactive Folium map centred on Paderborn.

In [13]:
!pip install ipywidgets -q

import ipywidgets as widgets
from IPython.display import display, HTML
import plotly.graph_objects as go


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: C:\Users\MoZa\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [14]:
from folium.plugins import HeatMap
import plotly.express as px

def build_linked_viz(filter_by="All hours",
                     show_fatal=True, show_serious=True, show_light=True,
                     show_schools=True, show_lights=True, show_heatmap=False):
    """Returns (folium_map, plotly_fig) — does NOT display anything itself."""

    selected_severities = []
    if show_fatal:   selected_severities.append("Fatal")
    if show_serious: selected_severities.append("Serious injury")
    if show_light:   selected_severities.append("Light injury")

    filtered = bike_acci.copy()
    if filter_by != "All hours":
        hour = int(filter_by.replace(":00", ""))
        filtered = filtered[filtered["USTUNDE"] == hour]
    if selected_severities:
        filtered = filtered[filtered["severity"].isin(selected_severities)]
    else:
        filtered = filtered.iloc[0:0]

    m = folium.Map(location=[51.718, 8.757], zoom_start=13)
    color_map = {"Fatal": "black", "Serious injury": "red", "Light injury": "orange"}

    if show_heatmap:
        HeatMap(bike_acci[["lat", "lon"]].values.tolist(), radius=15, blur=10, min_opacity=0.4).add_to(m)

    for _, row in filtered.iterrows():
        folium.CircleMarker(
            location=[row["lat"], row["lon"]], radius=6,
            color=color_map.get(row["severity"], "gray"), fill=True, fill_opacity=0.7,
            tooltip=f"{row['severity']} | {row['light']} | {row['weekday']} {row['USTUNDE']}:00"
        ).add_to(m)

    if show_schools:
        for _, row in df_schools.iterrows():
            folium.Marker(
                location=[row["lat"], row["lon"]], tooltip=row["name"],
                icon=folium.Icon(color="dark blue", icon="college", prefix="fa")
            ).add_to(m)

    if show_lights:
        for _, row in df_lights.iterrows():
            folium.CircleMarker(
                location=[row["lat"], row["lon"]], radius=3,
                color="#9b59b6", fill=True, fill_opacity=0.5, tooltip="Street lamp"
            ).add_to(m)

    legend_html = f"""
    <div style="position:fixed; bottom:30px; left:30px; z-index:1000;
         background:white; padding:12px; border-radius:8px;
         border:1px solid grey; font-size:13px; line-height:1.8;">
      <b>Paderborn Bike Accidents 2024</b><br>Showing: {len(filtered)} accidents<br><br>
      <span style="color:black">&#9679;</span> Fatal<br>
      <span style="color:red">&#9679;</span> Serious injury<br>
      <span style="color:orange">&#9679;</span> Light injury<br>
      <span style="color:blue">&#9679;</span> School<br>
      <span style="color:#9b59b6">&#9679;</span> Street lamp
    </div>"""
    m.get_root().html.add_child(folium.Element(legend_html))

    by_hour_sev = filtered.groupby(["USTUNDE", "severity"]).size().reset_index(name="count")
    fig = px.bar(
        by_hour_sev, x="USTUNDE", y="count", color="severity",
        labels={"USTUNDE": "Hour of Day", "count": "Accidents", "severity": "Severity"},
        title=f"Accidents by Hour & Severity — {filter_by}",
        color_discrete_map={"Light injury": "orange", "Serious injury": "red", "Fatal": "black"},
        barmode="stack"
    )
    fig.update_layout(xaxis=dict(tickmode="linear", tick0=0, dtick=1), height=350, margin=dict(t=40, b=40))

    return m, fig

In [15]:
available_hours = (
    ["All hours"] +
    [f"{h}:00" for h in sorted(bike_acci["USTUNDE"].unique())]
)

hour_dropdown = widgets.Dropdown(
    options=available_hours, value="All hours", description="Hour:",
    style={"description_width": "initial"}, layout=widgets.Layout(width="200px")
)
cb_fatal   = widgets.Checkbox(value=True,  description="Fatal",          indent=False)
cb_serious = widgets.Checkbox(value=True,  description="Serious injury",  indent=False)
cb_light   = widgets.Checkbox(value=True,  description="Light injury",    indent=False)
cb_schools  = widgets.Checkbox(value=True,  description="Schools",       indent=False)
cb_lights   = widgets.Checkbox(value=True,  description="Streetlights",  indent=False)
cb_heatmap  = widgets.Checkbox(value=False, description="Heatmap",       indent=False)

title        = widgets.HTML(value="<h3>Bicycle Accident Explorer — Paderborn 2024</h3>")
lbl_severity = widgets.HTML(value="<b>Severity:</b>")
lbl_layers   = widgets.HTML(value="<b>Layers:</b>")
out = widgets.Output()

def redraw(_=None):
    m, fig = build_linked_viz(
        filter_by    = hour_dropdown.value,
        show_fatal   = cb_fatal.value,
        show_serious = cb_serious.value,
        show_light   = cb_light.value,
        show_schools = cb_schools.value,
        show_lights  = cb_lights.value,
        show_heatmap = cb_heatmap.value,
    )
    out.clear_output(wait=True)
    with out:
        display(HTML(m._repr_html_()))
        display(fig)

for w in [hour_dropdown, cb_fatal, cb_serious, cb_light, cb_schools, cb_lights, cb_heatmap]:
    w.observe(redraw, names="value")

controls = widgets.HBox([
    widgets.VBox([hour_dropdown]),
    widgets.VBox([lbl_severity, cb_fatal, cb_serious, cb_light]),
    widgets.VBox([lbl_layers,   cb_schools, cb_lights, cb_heatmap]),
], layout=widgets.Layout(gap="30px", align_items="flex-start"))

redraw()
widgets.VBox([title, controls, out])

C:\Users\MoZa\AppData\Local\Temp\ipykernel_26604\1789184528.py:40: UserWarning: color argument of Icon should be one of: {'pink', 'white', 'cadetblue', 'red', 'blue', 'green', 'orange', 'gray', 'lightred', 'darkgreen', 'darkpurple', 'purple', 'lightgray', 'darkblue', 'darkred', 'lightblue', 'beige', 'black', 'lightgreen'}.
  icon=folium.Icon(color="dark blue", icon="college", prefix="fa")


In [16]:
import plotly.express as px

by_school = bike_acci.groupby("near_school").size().reset_index(name="count")
by_school["near_school"] = by_school["near_school"].map({
    True:  "Near a school (<200m)",
    False: "Not near a school"
})

fig_school = px.bar(
    by_school,
    x="near_school",
    y="count",
    labels={"near_school": "", "count": "Number of Accidents"},
    title="Bicycle Accidents — Near Schools vs. Elsewhere",
    color="near_school",
    color_discrete_map={
        "Near a school (<200m)": "#e74c3c",
        "Not near a school":     "#3498db"
    }
)
fig_school.update_layout(showlegend=False)
fig_school.show()

by_light = bike_acci.groupby("light").size().reset_index(name="count")
fig_light = px.bar(
    by_light,
    x="light",
    y="count",
    labels={"light": "Light Condition", "count": "Number of Accidents"},
    title="Bicycle Accidents in Paderborn 2024 — by Light Condition",
    color="light",
    color_discrete_map={"Daylight": "#f4c542", "Dusk/dawn": "#f48c42", "Dark": "#2c2c54"}
)
fig_light.show()

weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
by_day = bike_acci.groupby("weekday").size().reset_index(name="count")
by_day["weekday"] = pd.Categorical(by_day["weekday"], categories=weekday_order, ordered=True)
by_day = by_day.sort_values("weekday")
fig_day = px.bar(
    by_day, x="weekday", y="count",
    labels={"weekday": "Day of Week", "count": "Number of Accidents"},
    title="Bicycle Accidents in Paderborn 2024 — by Day of Week",
    color="count", color_continuous_scale="Reds"
)
fig_day.update_layout(coloraxis_showscale=False)
fig_day.show()

by_hour = bike_acci.groupby("USTUNDE").size().reset_index(name="count")
fig_hour = px.bar(
    by_hour, x="USTUNDE", y="count",
    labels={"USTUNDE": "Hour of Day", "count": "Number of Accidents"},
    title="Bicycle Accidents in Paderborn 2024 — by Hour of Day",
    color="count", color_continuous_scale="Reds"
)
fig_hour.update_layout(xaxis=dict(tickmode="linear", tick0=0, dtick=1), coloraxis_showscale=False)
fig_hour.show()

by_hour_sev = bike_acci.groupby(["USTUNDE", "severity"]).size().reset_index(name="count")
fig_sev = px.bar(
    by_hour_sev, x="USTUNDE", y="count", color="severity",
    labels={"USTUNDE": "Hour of Day", "count": "Accidents", "severity": "Severity"},
    title="Bicycle Accidents by Hour and Severity — Paderborn 2024",
    color_discrete_map={"Light injury": "orange", "Serious injury": "red", "Fatal": "black"},
    barmode="stack"
)
fig_sev.update_layout(xaxis=dict(tickmode="linear", tick0=0, dtick=1))
fig_sev.show()